# Visualization 2 - Regional Name Heatmap

This notebook creates a simple heatmap for the second mini-project question:

- Are some names more popular in certain places?
- Are names that are popular nationally also popular everywhere?
- Can we see geographic differences in naming?

The heatmap compares a few **signature names** across a few **signature departments**, plus one extra column for **France**.

In [82]:
import pandas as pd
import geopandas as gpd
import altair as alt

alt.data_transformers.disable_max_rows()
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [83]:
# Load and clean the department-level baby names dataset.
names = pd.read_csv('dpt2020.csv', sep=';')
names = names[(names['preusuel'] != '_PRENOMS_RARES') & (names['dpt'] != 'XX') & (names['annais'] != 'XXXX')].copy()
names['annais'] = names['annais'].astype(int)
names['nombre'] = names['nombre'].astype(int)
names['dpt'] = names['dpt'].astype(str).str.zfill(2)

depts = gpd.read_file('departements-version-simplifiee.geojson')[['code', 'nom']]

# Focus on the recent period to keep the comparison concrete and readable.
recent = names[names['annais'].between(2010, 2020)].copy()

# A few signature names from the recent national top names.
selected_names = ['GABRIEL', 'LUCAS', 'EMMA', 'LOUIS', 'JADE', 'NATHAN', 'LOUISE', 'LÉO']
top_name_pool = (
    recent.groupby('preusuel', as_index=False)['nombre']
    .sum()
    .sort_values('nombre', ascending=False)
    .head(150)
)
available_names = top_name_pool['preusuel'].tolist()

# A few signature departments with large birth volumes and different contexts.
selected_depts = ['75', '59', '69', '13', '93', '33', '44', '31']
dept_labels = depts.assign(region_label=lambda d: d['nom'] + ' (' + d['code'] + ')')[['code', 'region_label']]
selected_dept_labels = dept_labels[dept_labels['code'].isin(selected_depts)].copy()

# Regional name counts and total births by department.
dept_name = (
    recent[recent['preusuel'].isin(available_names)]
    .groupby(['dpt', 'preusuel'], as_index=False)['nombre']
    .sum()
)

dept_total = (
    recent
    .groupby('dpt', as_index=False)['nombre']
    .sum()
    .rename(columns={'nombre': 'total_births'})
)

dept_heat = (
    dept_name.merge(dept_total, on='dpt', how='left')
    .merge(dept_labels, left_on='dpt', right_on='code', how='left')
)
dept_heat['share'] = dept_heat['nombre'] / dept_heat['total_births']
dept_heat = dept_heat[['region_label', 'preusuel', 'nombre', 'total_births', 'share']]

# National reference column.
fr_name = (
    recent[recent['preusuel'].isin(available_names)]
    .groupby('preusuel', as_index=False)['nombre']
    .sum()
)
fr_total = recent['nombre'].sum()
fr_heat = fr_name.copy()
fr_heat['region_label'] = 'France'
fr_heat['total_births'] = fr_total
fr_heat['share'] = fr_heat['nombre'] / fr_heat['total_births']
fr_heat = fr_heat[['region_label', 'preusuel', 'nombre', 'total_births', 'share']]

heatmap_data = pd.concat([fr_heat, dept_heat], ignore_index=True)

name_order = selected_names[::-1]
region_order = ['France'] + selected_dept_labels['region_label'].tolist()
all_region_order = ['France'] + dept_labels['region_label'].tolist()
department_selector_data = pd.DataFrame({'region_label': dept_labels['region_label'].tolist()})

heatmap_data.head()

,region_label,preusuel,nombre,total_births,share
0,France,AARON,22869,6015739,0.003802
1,France,ADAM,44387,6015739,0.007378
2,France,ADRIEN,10785,6015739,0.001793
3,France,ADÈLE,12739,6015739,0.002118
4,France,AGATHE,17315,6015739,0.002878


In [84]:
import ipywidgets as widgets
from IPython.display import display

selected_region_labels = selected_dept_labels['region_label'].tolist()[:4].copy()
selected_name_labels = selected_names[:5].copy()
all_region_labels = dept_labels['region_label'].tolist()
all_name_labels = available_names.copy()
chart_handle = None

def build_heatmap(selected_regions, selected_names_for_chart):
    visible_regions = ['France'] + selected_regions
    visible_data = heatmap_data[
        heatmap_data['region_label'].isin(visible_regions)
        & heatmap_data['preusuel'].isin(selected_names_for_chart)
    ].copy()

    base = alt.Chart(visible_data).transform_calculate(
        region_order="datum.region_label === 'France' ? 0 : indexof(" + str(selected_regions) + ", datum.region_label) + 1"
    )

    chart = base.mark_rect().encode(
        x=alt.X('region_label:N', title='Region / department', sort=visible_regions),
        y=alt.Y('preusuel:N', title='Name', sort=selected_names_for_chart[::-1]),
        color=alt.Color(
            'share:Q',
            title='Share of births (2010-2020)',
            scale=alt.Scale(scheme='oranges'),
            legend=alt.Legend(format='.1%')
        ),
        tooltip=[
            alt.Tooltip('region_label:N', title='Region'),
            alt.Tooltip('preusuel:N', title='Name'),
            alt.Tooltip('nombre:Q', title='Births with this name'),
            alt.Tooltip('total_births:Q', title='Total births'),
            alt.Tooltip('share:Q', title='Share', format='.3%')
        ]
    ).properties(
        width=760,
        height=320,
        title='Some Baby Names Are More Regional Than Others: Departments vs France (2010-2020)'
    )

    labels = base.mark_text(fontSize=10).encode(
        x=alt.X('region_label:N', sort=visible_regions),
        y=alt.Y('preusuel:N', sort=selected_names_for_chart[::-1]),
        text=alt.Text('share:Q', format='.2%'),
        color=alt.condition('datum.share > 0.010', alt.value('white'), alt.value('#222'))
    )

    return chart + labels

def refresh_controls():
    add_region_dropdown.options = [label for label in all_region_labels if label not in selected_region_labels]
    remove_region_dropdown.options = selected_region_labels.copy()
    add_name_dropdown.options = [label for label in all_name_labels if label not in selected_name_labels]
    remove_name_dropdown.options = selected_name_labels.copy()
    if add_region_dropdown.options:
        add_region_dropdown.value = add_region_dropdown.options[0]
    else:
        add_region_dropdown.value = None
    if remove_region_dropdown.options:
        remove_region_dropdown.value = remove_region_dropdown.options[0]
    else:
        remove_region_dropdown.value = None
    if add_name_dropdown.options:
        add_name_dropdown.value = add_name_dropdown.options[0]
    else:
        add_name_dropdown.value = None
    if remove_name_dropdown.options:
        remove_name_dropdown.value = remove_name_dropdown.options[0]
    else:
        remove_name_dropdown.value = None

def refresh_chart():
    chart_handle.update(build_heatmap(selected_region_labels, selected_name_labels))

def on_add_region_clicked(_):
    value = add_region_dropdown.value
    if value and value not in selected_region_labels:
        selected_region_labels.append(value)
        refresh_controls()
        refresh_chart()

def on_remove_region_clicked(_):
    value = remove_region_dropdown.value
    if value and value in selected_region_labels:
        selected_region_labels.remove(value)
        refresh_controls()
        refresh_chart()

def on_add_name_clicked(_):
    value = add_name_dropdown.value
    if value and value not in selected_name_labels:
        selected_name_labels.append(value)
        refresh_controls()
        refresh_chart()

def on_remove_name_clicked(_):
    value = remove_name_dropdown.value
    if value and value in selected_name_labels:
        selected_name_labels.remove(value)
        refresh_controls()
        refresh_chart()

add_region_dropdown = widgets.Dropdown(
    options=[],
    description='Add region:',
    layout=widgets.Layout(width='340px')
)

remove_region_dropdown = widgets.Dropdown(
    options=[],
    description='Remove region:',
    layout=widgets.Layout(width='340px')
)

add_name_dropdown = widgets.Dropdown(
    options=[],
    description='Add name:',
    layout=widgets.Layout(width='340px')
)

remove_name_dropdown = widgets.Dropdown(
    options=[],
    description='Remove name:',
    layout=widgets.Layout(width='340px')
)

add_region_button = widgets.Button(description='Add', button_style='primary')
remove_region_button = widgets.Button(description='Remove')
add_name_button = widgets.Button(description='Add', button_style='primary')
remove_name_button = widgets.Button(description='Remove')
add_region_button.on_click(on_add_region_clicked)
remove_region_button.on_click(on_remove_region_clicked)
add_name_button.on_click(on_add_name_clicked)
remove_name_button.on_click(on_remove_name_clicked)

refresh_controls()
display(widgets.VBox([
    widgets.HBox([add_region_dropdown, add_region_button, remove_region_dropdown, remove_region_button]),
    widgets.HBox([add_name_dropdown, add_name_button, remove_name_dropdown, remove_name_button]),
]))
chart_handle = display(build_heatmap(selected_region_labels, selected_name_labels), display_id=True)


alt.LayerChart(...)

## Why this works for Visualization 2


### Advantages

- It compares each department with **France** directly.
- The heatmap gives a fast overview of regional variation across several names at once.
- It uses **relative popularity** instead of raw counts, so large departments do not automatically dominate.
- Comparing departments to the national baseline makes local overrepresentation easier to interpret.
- Darker or lighter cells immediately show where a name is more common or less common.
- The matrix layout supports side-by-side comparison better than looking at one map at a time.

### Disadvantages

- The heatmap is less geographically intuitive than a full choropleth map.
- Department names or codes may be harder to recognize without prior knowledge of France.
- Showing only a selected set of names limits discovery of unexpected regional names.